# PS S6E4 - exp_20260408_023_cat_pairwise_te_topk_reduction

目的:
 - exp_20260407_018_cat_pairwise_te_bias_from_shared_min の pairwise 採用本数を削減
 - pairwise が本体であることは 021 で確認済み
 - 今回は「135本全部が必要なのか」を切る

方針:
 - 018 の構成を基本維持
 - pairwise combinations は元通り作る
 - cardinality filter 後の候補から top-k だけ採用
 - top-k はまず 64 本で確認
 - original / bias / physical / TE_ORIG は維持

## Imports

In [1]:
import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd

from itertools import combinations
from pathlib import Path

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from scipy.special import logit

import torch

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    EXP_ID = "exp_20260408_023_cat_pairwise_te_topk_reduction"
    EXP_NAME = "cat_pairwise_te_topk_reduction"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    SEED = 42
    N_SPLITS = 5
    TE_INNER_CV = 3

    ORIG_ROW_WEIGHT = 0.35

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {v: k for k, v in TARGET_MAP.items()}

    CATS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Mulching_Used",
        "Region",
    ]

    NUMS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    ALL_BASE = NUMS + CATS

    CATBOOST_PARAMS = {
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 8,
        "grow_policy": "SymmetricTree",
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "random_seed": SEED,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
        "devices": "0" if torch.cuda.is_available() else None,
        "verbose": 0,
        "early_stopping_rounds": 150,
    }

    BIAS_STEPS = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]
    LOGIT_EPS = 1e-15

    FEATURE_IMPORTANCE_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/feature_importance.csv"
    TOPK_PAIRWISE = 64

print(CFG.SAVE_DIR)

/kaggle/working/exp_20260408_023_cat_pairwise_te_topk_reduction


## load data

In [3]:
COMP_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")
ORIG_PATH = Path("/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv")

train_raw = pd.read_csv(COMP_DIR / "train.csv").dropna(subset=[CFG.TARGET_COL]).reset_index(drop=True)
test_raw = pd.read_csv(COMP_DIR / "test.csv").reset_index(drop=True)
orig_raw = pd.read_csv(ORIG_PATH).reset_index(drop=True)

print(f"train_raw: {train_raw.shape}")
print(f"test_raw : {test_raw.shape}")
print(f"orig_raw : {orig_raw.shape}")
display(train_raw.head())

train_raw: (630000, 21)
test_raw : (270000, 20)
orig_raw : (10000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## FE Helpers

In [4]:
def add_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ET_proxy"] = (d["Temperature_C"] * d["Wind_Speed_kmh"] * d["Sunlight_Hours"]) / (d["Humidity"] + 1.0)
    d["water_deficit"] = d["Soil_Moisture"] - (d["Rainfall_mm"] + d["Previous_Irrigation_mm"]) * 0.1
    d["soil_quality"] = d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)
    return d


def build_orig_target_prior_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    base_cols: list[str],
    target_col: str,
    target_map: dict,
):
    train_out = train_df.copy()
    test_out = test_df.copy()
    orig_out = orig_df.copy()

    orig_target_numeric = orig_out[target_col].map(target_map).astype(np.float32)

    for col in base_cols:
        feat_name = f"TE_ORIG_{col}"

        if orig_out[col].dtype == "object":
            mapping = orig_target_numeric.groupby(orig_out[col]).mean().to_dict()
            fallback = float(orig_target_numeric.mean())

            train_out[feat_name] = train_out[col].map(mapping).fillna(fallback).astype(np.float32)
            test_out[feat_name] = test_out[col].map(mapping).fillna(fallback).astype(np.float32)
            orig_out[feat_name] = orig_out[col].map(mapping).fillna(fallback).astype(np.float32)

        else:
            edges = np.histogram_bin_edges(orig_out[col].dropna(), bins=10)
            bucket_means = orig_target_numeric.groupby(
                pd.cut(orig_out[col], bins=edges, include_lowest=True)
            ).mean().values

            fallback = float(orig_target_numeric.mean())

            def map_bins(v):
                if pd.isna(v):
                    return fallback
                idx = np.digitize(v, edges) - 1
                idx = min(max(idx, 0), len(bucket_means) - 1)
                val = bucket_means[idx]
                if pd.isna(val):
                    return fallback
                return float(val)

            train_out[feat_name] = train_out[col].apply(map_bins).astype(np.float32)
            test_out[feat_name] = test_out[col].apply(map_bins).astype(np.float32)
            orig_out[feat_name] = orig_out[col].apply(map_bins).astype(np.float32)

    return train_out, test_out, orig_out


def build_pairwise_combinations(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    source_cols: list[str],
):
    train_out = train_df.copy()
    test_out = test_df.copy()
    orig_out = orig_df.copy()

    pair_cols = []
    raw_pair_count = 0

    for left, right in combinations(source_cols, 2):
        raw_pair_count += 1
        feat_name = f"{left}___{right}"

        tr_token = train_out[left].astype(str) + "_" + train_out[right].astype(str)
        te_token = test_out[left].astype(str) + "_" + test_out[right].astype(str)
        og_token = orig_out[left].astype(str) + "_" + orig_out[right].astype(str)

        combined = pd.concat([tr_token, te_token, og_token], ignore_index=True)
        codes, _ = pd.factorize(combined)

        if pd.Series(codes).nunique() <= (len(combined) // 2):
            train_out[feat_name] = codes[:len(train_out)].astype(np.int32)
            test_out[feat_name] = codes[len(train_out):len(train_out) + len(test_out)].astype(np.int32)
            orig_out[feat_name] = codes[len(train_out) + len(test_out):].astype(np.int32)
            pair_cols.append(feat_name)

    return train_out, test_out, orig_out, pair_cols, raw_pair_count

## make features

In [5]:
train = add_physical_features(train_raw)
test = add_physical_features(test_raw)
orig = add_physical_features(orig_raw)

train, test, orig = build_orig_target_prior_features(
    train_df=train,
    test_df=test,
    orig_df=orig,
    base_cols=CFG.ALL_BASE,
    target_col=CFG.TARGET_COL,
    target_map=CFG.TARGET_MAP,
)

train, test, orig, pair_cols, raw_pair_count = build_pairwise_combinations(
    train_df=train,
    test_df=test,
    orig_df=orig,
    source_cols=CFG.ALL_BASE,
)

for c in CFG.CATS:
    train[c] = train[c].astype(str)
    test[c] = test[c].astype(str)
    orig[c] = orig[c].astype(str)

y_train_full = train[CFG.TARGET_COL].map(CFG.TARGET_MAP).values.astype(int)
y_orig_full = orig[CFG.TARGET_COL].map(CFG.TARGET_MAP).values.astype(int)

print(f"raw_pair_count      = {raw_pair_count}")
print(f"accepted_pair_count = {len(pair_cols)}")
print(pair_cols[:10])

raw_pair_count      = 171
accepted_pair_count = 135
['Soil_pH___Organic_Carbon', 'Soil_pH___Electrical_Conductivity', 'Soil_pH___Temperature_C', 'Soil_pH___Sunlight_Hours', 'Soil_pH___Wind_Speed_kmh', 'Soil_pH___Field_Area_hectare', 'Soil_pH___Soil_Type', 'Soil_pH___Crop_Type', 'Soil_pH___Crop_Growth_Stage', 'Soil_pH___Season']


## pair importance を feature_importance.csv から復元

In [6]:
fi_018 = pd.read_csv(CFG.FEATURE_IMPORTANCE_PATH)
display(fi_018.head())

pair_rows = fi_018[fi_018["feature"].astype(str).str.startswith("TE_pair_")].copy()
pair_rows["te_idx"] = pair_rows["feature"].str.replace("TE_pair_", "", regex=False).astype(int)

# multiclass TargetEncoder により、1 pair -> 3列
pair_rows["pair_idx"] = pair_rows["te_idx"] // 3

pair_importance_df = (
    pair_rows.groupby("pair_idx", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

pair_importance_df["pair_name"] = pair_importance_df["pair_idx"].map(lambda i: pair_cols[i])

print("n_pair_rows in fi:", len(pair_rows))
print("n_pair_groups    :", len(pair_importance_df))
display(pair_importance_df.head(20))

,feature,importance
0,TE_pair_51,6.173397
1,Soil_Moisture,4.236748
2,Rainfall_mm,4.123363
3,Wind_Speed_kmh,3.282644
4,TE_pair_267,2.688973


n_pair_rows in fi: 405
n_pair_groups    : 135


,pair_idx,importance,pair_name
0,17,9.478990,Soil_Moisture___Crop_Growth_Stage
1,89,4.472910,Wind_Speed_kmh___Mulching_Used
2,21,3.103438,Soil_Moisture___Mulching_Used
3,72,2.775617,Rainfall_mm___Mulching_Used
4,68,2.676290,Rainfall_mm___Crop_Growth_Stage
5,56,2.572612,Temperature_C___Mulching_Used
6,123,2.517545,Crop_Growth_Stage___Mulching_Used
7,52,2.466225,Temperature_C___Crop_Growth_Stage
8,85,2.434662,Wind_Speed_kmh___Crop_Growth_Stage
9,69,0.979889,Rainfall_mm___Season


## select TopK pairs

In [7]:
selected_pair_idx = pair_importance_df.head(CFG.TOPK_PAIRWISE)["pair_idx"].tolist()
selected_pair_cols = [pair_cols[i] for i in selected_pair_idx]

print("TOPK_PAIRWISE =", CFG.TOPK_PAIRWISE)
print("selected_pair_count =", len(selected_pair_cols))
print(selected_pair_cols[:20])

TOPK_PAIRWISE = 64
selected_pair_count = 64
['Soil_Moisture___Crop_Growth_Stage', 'Wind_Speed_kmh___Mulching_Used', 'Soil_Moisture___Mulching_Used', 'Rainfall_mm___Mulching_Used', 'Rainfall_mm___Crop_Growth_Stage', 'Temperature_C___Mulching_Used', 'Crop_Growth_Stage___Mulching_Used', 'Temperature_C___Crop_Growth_Stage', 'Wind_Speed_kmh___Crop_Growth_Stage', 'Rainfall_mm___Season', 'Rainfall_mm___Irrigation_Type', 'Organic_Carbon___Crop_Growth_Stage', 'Previous_Irrigation_mm___Mulching_Used', 'Humidity___Crop_Growth_Stage', 'Soil_Moisture___Water_Source', 'Electrical_Conductivity___Crop_Growth_Stage', 'Soil_pH___Crop_Growth_Stage', 'Rainfall_mm___Water_Source', 'Sunlight_Hours___Crop_Growth_Stage', 'Previous_Irrigation_mm___Crop_Growth_Stage']


## CV with TE on selected pairwise only

In [8]:
skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_probs = np.zeros((len(train), 3), dtype=np.float32)
test_probs_sum = np.zeros((len(test), 3), dtype=np.float32)
fold_scores_raw = []
importances = []
final_feature_names = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y_train_full), 1):
    X_tr_fold = train.iloc[tr_idx].copy()
    y_tr_fold = y_train_full[tr_idx]

    X_va_fold = train.iloc[va_idx].copy()
    y_va_fold = y_train_full[va_idx]

    # original は outer train 側にだけ merge
    X_tr_merged = pd.concat([X_tr_fold, orig], axis=0).reset_index(drop=True)
    y_tr_merged = np.concatenate([y_tr_fold, y_orig_full])

    encoder = TargetEncoder(
        target_type="multiclass",
        cv=CFG.TE_INNER_CV,
        random_state=CFG.SEED,
    )

    enc_tr = encoder.fit_transform(X_tr_merged[selected_pair_cols], y_tr_merged).astype(np.float32)
    enc_va = encoder.transform(X_va_fold[selected_pair_cols]).astype(np.float32)
    enc_te = encoder.transform(test[selected_pair_cols]).astype(np.float32)

    X_tr_final = X_tr_merged.drop(columns=pair_cols).copy()
    X_va_final = X_va_fold.drop(columns=pair_cols).copy()
    X_te_final = test.drop(columns=pair_cols).copy()

    te_col_names = [f"TE_pair_{i}" for i in range(enc_tr.shape[1])]
    X_tr_final[te_col_names] = enc_tr
    X_va_final[te_col_names] = enc_va
    X_te_final[te_col_names] = enc_te

    # target と id を除外
    X_tr_final = X_tr_final.drop(columns=[CFG.TARGET_COL, CFG.ID_COL], errors="ignore")
    X_va_final = X_va_final.drop(columns=[CFG.TARGET_COL, CFG.ID_COL], errors="ignore")
    X_te_final = X_te_final.drop(columns=[CFG.TARGET_COL, CFG.ID_COL], errors="ignore")

    if final_feature_names is None:
        final_feature_names = X_tr_final.columns.tolist()

    cat_cols_final = [c for c in CFG.CATS if c in X_tr_final.columns]

    sw = compute_sample_weight("balanced", y_tr_merged).astype(np.float32)
    sw[len(tr_idx):] *= CFG.ORIG_ROW_WEIGHT

    params = CFG.CATBOOST_PARAMS.copy()
    if not torch.cuda.is_available():
        params.pop("devices", None)

    model = CatBoostClassifier(**params)
    model.fit(
        Pool(X_tr_final, y_tr_merged, cat_features=cat_cols_final, weight=sw),
        eval_set=Pool(X_va_final, y_va_fold, cat_features=cat_cols_final),
    )

    oof_probs[va_idx] = model.predict_proba(X_va_final)
    test_probs_sum += model.predict_proba(X_te_final)
    importances.append(model.get_feature_importance())

    fold_ba = balanced_accuracy_score(y_va_fold, oof_probs[va_idx].argmax(axis=1))
    fold_scores_raw.append(float(fold_ba))
    print(f"fold {fold}/{CFG.N_SPLITS} raw_ba = {fold_ba:.6f}")

    del X_tr_final, X_va_final, X_te_final, enc_tr, enc_va, enc_te, model
    gc.collect()

raw_cv_ba = float(balanced_accuracy_score(y_train_full, oof_probs.argmax(axis=1)))
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

fold 1/5 raw_ba = 0.976371
fold 2/5 raw_ba = 0.976825
fold 3/5 raw_ba = 0.978974
fold 4/5 raw_ba = 0.977072
fold 5/5 raw_ba = 0.976819
raw_cv_ba = 0.977212


## bias tuning

In [9]:
def get_tuned_preds(probs: np.ndarray, bias: np.ndarray) -> np.ndarray:
    adjusted = logit(np.clip(probs, CFG.LOGIT_EPS, 1.0 - CFG.LOGIT_EPS)) + bias
    return np.argmax(adjusted, axis=1)


best_bias = np.zeros(3, dtype=np.float32)
best_score = raw_cv_ba

for step in CFG.BIAS_STEPS:
    improved = True
    while improved:
        improved = False
        for ci in range(3):
            for direction in [1, -1]:
                trial_bias = best_bias.copy()
                trial_bias[ci] += direction * step

                score = balanced_accuracy_score(
                    y_train_full,
                    get_tuned_preds(oof_probs, trial_bias)
                )

                if score > best_score + 1e-9:
                    best_score = float(score)
                    best_bias = trial_bias.copy()
                    improved = True

print("best_bias =", best_bias.tolist())
print(f"biased_cv_ba = {best_score:.6f}")
print(f"improvement  = {best_score - raw_cv_ba:.6f}")

best_bias = [-1.0799999237060547, -1.1999999284744263, 1.0]
biased_cv_ba = 0.978898
improvement  = 0.001686


## save artifacts

In [10]:
def get_calibrated_probs(probs: np.ndarray, bias: np.ndarray) -> np.ndarray:
    log_p = logit(np.clip(probs, CFG.LOGIT_EPS, 1.0 - CFG.LOGIT_EPS)) + bias
    exp_p = np.exp(log_p)
    return exp_p / exp_p.sum(axis=1, keepdims=True)


test_probs = test_probs_sum / CFG.N_SPLITS

oof_probs_biased = get_calibrated_probs(oof_probs, best_bias)
test_probs_biased = get_calibrated_probs(test_probs, best_bias)

oof_pred_raw = np.argmax(oof_probs, axis=1)
oof_pred_biased = np.argmax(oof_probs_biased, axis=1)
test_pred_biased = np.argmax(test_probs_biased, axis=1)

submission = pd.DataFrame({
    CFG.ID_COL: test_raw[CFG.ID_COL],
    CFG.TARGET_COL: pd.Series(test_pred_biased).map(CFG.INV_TARGET_MAP),
})
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_probs)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_probs)
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_probs_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", test_probs_biased)
np.save(CFG.SAVE_DIR / "best_class_bias.npy", best_bias)

feature_columns_df = pd.DataFrame({
    "feature": final_feature_names,
    "is_cat_feature": [c in CFG.CATS for c in final_feature_names],
    "is_pair_te_feature": [str(c).startswith("TE_pair_") for c in final_feature_names],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

avg_importance = np.mean(importances, axis=0)
fi_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": avg_importance,
}).sort_values("importance", ascending=False)
fi_df.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)

pair_importance_df.to_csv(CFG.SAVE_DIR / "pair_importance_from_018.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "raw_cv_ba": raw_cv_ba,
            "biased_cv_ba": best_score,
            "fold_scores_raw": fold_scores_raw,
            "best_bias": best_bias.tolist(),
            "orig_row_weight": CFG.ORIG_ROW_WEIGHT,
            "raw_pair_count": raw_pair_count,
            "accepted_pair_count": len(pair_cols),
            "selected_pair_count": len(selected_pair_cols),
            "topk_pairwise": CFG.TOPK_PAIRWISE,
            "selected_pair_cols": selected_pair_cols,
            "n_final_features": len(final_feature_names),
            "n_pair_te_features": int(sum(str(c).startswith("TE_pair_") for c in final_feature_names)),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260408_023_cat_pairwise_te_topk_reduction


## summary

In [11]:
summary_df = pd.DataFrame({
    "item": [
        "raw_pair_count",
        "accepted_pair_count",
        "selected_pair_count",
        "topk_pairwise",
        "raw_cv_ba",
        "biased_cv_ba",
        "bias_low",
        "bias_medium",
        "bias_high",
        "n_final_features",
    ],
    "value": [
        raw_pair_count,
        len(pair_cols),
        len(selected_pair_cols),
        CFG.TOPK_PAIRWISE,
        raw_cv_ba,
        best_score,
        best_bias[0],
        best_bias[1],
        best_bias[2],
        len(final_feature_names),
    ],
})
display(summary_df)
display(pair_importance_df.head(20))

,item,value
0,raw_pair_count,171.000000
1,accepted_pair_count,135.000000
2,selected_pair_count,64.000000
3,topk_pairwise,64.000000
4,raw_cv_ba,0.977212
5,biased_cv_ba,0.978898
6,bias_low,-1.080000
7,bias_medium,-1.200000
8,bias_high,1.000000
9,n_final_features,233.000000


,pair_idx,importance,pair_name
0,17,9.478990,Soil_Moisture___Crop_Growth_Stage
1,89,4.472910,Wind_Speed_kmh___Mulching_Used
2,21,3.103438,Soil_Moisture___Mulching_Used
3,72,2.775617,Rainfall_mm___Mulching_Used
4,68,2.676290,Rainfall_mm___Crop_Growth_Stage
5,56,2.572612,Temperature_C___Mulching_Used
6,123,2.517545,Crop_Growth_Stage___Mulching_Used
7,52,2.466225,Temperature_C___Crop_Growth_Stage
8,85,2.434662,Wind_Speed_kmh___Crop_Growth_Stage
9,69,0.979889,Rainfall_mm___Season
